# 📊 Proyecto Final: Análisis de Datos y Dashboard de Ventas en Retail
**Universidad Central de Venezuela (UCV)**

**Escuela:** Estadística

**Materia:** Computación

**Autores:** Radames Montiel y Rafael Ferrer

**Profesores:** Jesus Ochoa y Oliver Triveño

**Año:** 2026

---

**Resumen Ejecutivo:**
Este cuaderno de Jupyter documenta el proceso completo análisis exploratorio de datos aplicado a una base de datos de trensacciones.

Este proyecto ha sido delimitado en los siguientes pasos:
1. Extracción, Transformación y Carga.
3. Análisis Exploratorio de Datos respondiendo a 5 objetivos de negocio.
4. Despliegue de un Dashboard interactivo usando Streamlit.

# Extracción, Transformación y Carga (ETL).

ETL, que básicamente es preparar los datos de la tabla completamente limpios para que el análisis sea eficaz.

Las acciones estructuradas que se ejecutarán en este bloque son:


1. **Extracción:** Importo mis librerías y luego uso pandas para abrir el archivo original (.csv) y cargarlo aquí en el cuaderno para poder trabajar con él.
2. **Ver qué hay en la tabla:** Voy a revisar si los datos están completos. Me fijaré si hay celdas vacías (nulos), si hay filas repetidas y qué tipo de información tiene cada columna (si son números o texto).
3. **Transformación :** Voy a borrar las columnas de Transaction ID y Customer ID. Como son solo números de identificación únicos para cada compra, no me sirven para analizar las tendencias o los grupos de clientes.
4. **Carga:** Una vez que ya borré lo que no servía y verifiqué que todo está en orden, guardo esta versión nueva en un archivo llamado Retail_Limpio.csv.



In [1]:
#Importamos nuestras librerías

import pandas as pd
import plotly.express as px
import streamlit as st

In [2]:
#Procedemos a leer nuestro dataframe y le asignamos el valor df. Procedemos a revisar la cantidad de filas y columnas.

df = pd.read_csv('retail.csv')

print("--- Filas y Columnas ---")

df.head()




--- Filas y Columnas ---


,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100


In [3]:
# Procedemos a ver los tipos de datos de las columnas

print("--- TIPOS DE DATOS ---")
print(df.info())
print()

print("--- FILAS Y COLUMNAS ---")
print(df.shape)



--- TIPOS DE DATOS ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    1000 non-null   int64 
 1   Date              1000 non-null   object
 2   Customer ID       1000 non-null   object
 3   Gender            1000 non-null   object
 4   Age               1000 non-null   int64 
 5   Product Category  1000 non-null   object
 6   Quantity          1000 non-null   int64 
 7   Price per Unit    1000 non-null   int64 
 8   Total Amount      1000 non-null   int64 
dtypes: int64(5), object(4)
memory usage: 70.4+ KB
None

--- FILAS Y COLUMNAS ---
(1000, 9)


In [4]:
# Extraemos estadísticas descriptivas de nuestro Dataframe
df.describe().round(2)

,Transaction ID,Age,Quantity,Price per Unit,Total Amount
count,1000.00,1000.00,1000.00,1000.00,1000.0
mean,500.50,41.39,2.51,179.89,456.0
std,288.82,13.68,1.13,189.68,560.0
min,1.00,18.00,1.00,25.00,25.0
25%,250.75,29.00,1.00,30.00,60.0
50%,500.50,42.00,3.00,50.00,135.0
75%,750.25,53.00,4.00,300.00,900.0
max,1000.00,64.00,4.00,500.00,2000.0


In [5]:
#Buscamos filas duplicadas o datos vacíos

print("--- DATOS VACÍOS ---")
print(df.isnull().sum())

print("\n--- FILAS DUPLICADAS ---")
print(df.duplicated().sum())

--- DATOS VACÍOS ---
Transaction ID      0
Date                0
Customer ID         0
Gender              0
Age                 0
Product Category    0
Quantity            0
Price per Unit      0
Total Amount        0
dtype: int64

--- FILAS DUPLICADAS ---
0


In [6]:
# Tras verificar que nuestro dataset tiene 0 valores nulos y 0 duplicados, procedemos a eliminar las variables `Customer ID` y `Transaction ID` al ser consideradas no relevantes y no aportar valor estadístico para nuestros análisis de agrupación demográfica

df.drop(columns=['Customer ID', 'Transaction ID'], inplace=True)

# Vemos nuestros datos nuevamente
df.head()



,Date,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
0,2023-11-24,Male,34,Beauty,3,50,150
1,2023-02-27,Female,26,Clothing,2,500,1000
2,2023-01-13,Male,50,Electronics,1,30,30
3,2023-05-21,Male,37,Clothing,1,500,500
4,2023-05-06,Male,30,Beauty,2,50,100


In [7]:
#Procedemos a crear el archivo de datos limpios

df.to_csv('Retail_Limpio.csv', index=False)

# FIJAMOS NUESTROS OBJETIVOS

## Impacto de las Caracteristicas del cliente en la Facturación

### Hipotesis: Demostraremos si el perfil demográfico de los clientes influye significativamente en el volumen de facturación y si existen segmentos específicos que generan un mayor ticket promedio que otros.


# Objetivo 1: Determinar si existen diferencias significativas en el ticket promedio entre clientes hombres y mujeres



**Justificación Metodológica:**
Aquí lo que voy a hacer es agrupar a los clientes por género para ver cómo se comportan. Hemos organizado el proceso en pasos de fácil entendimiento:

1. Contar cuántos registros tengo de cada uno.
2. Calcular el promedio de la columna 'Total Amount' para hombres y para mujeres.
3. Dibujar una gráfica para que sea más fácil de entender.
4. Finalmente se presenta la conclusion acerca de que nos arrojó el manejo de datos y nuestro gráfico. 

In [8]:
# Arreglamos nuestros datos para hombres y mujeres

print("--- Total por género ---")
print(df['Gender'].value_counts())

print()
ticket_genero = df.groupby('Gender')['Total Amount'].mean().round(2).reset_index()
print("--- Ticket Promedio por Género ---")
print(ticket_genero)


--- Total por género ---
Gender
Female    510
Male      490
Name: count, dtype: int64

--- Ticket Promedio por Género ---
   Gender  Total Amount
0  Female        456.55
1    Male        455.43


In [9]:
# Gráfico de barras simple
fig1 = px.bar(ticket_genero, 
              x='Gender', 
              y='Total Amount', 
              color='Gender',          
              text_auto='.2f',          
              title='Gasto Promedio por Género',
              color_discrete_map={'Female': 'lightpink', 'Male': 'darkblue'})
fig1.show()

#### Tras analizar las transacciones, descubrimos que el ticket promedio no varía significativamente entre géneros (Mujeres: 456.55 vs Hombres: 455.43). La diferencia es de apenas 1.12. Esto nos indica que el género no es un factor determinante en el monto gastado por visita. Ambos grupos tienen una disposición a gastar prácticamente igual.

# Objetivo 2: Analizar la relación entre la edad del cliente y la categoría de producto elegida

**Justificación Metodológica:** 
Para evaluar el comportamiento de consumo basado en la edad, la variable numérica continua `Age` presenta demasiada dispersión. Para que sea más intuitivo de trabajarla, voy a agrupar a las personas en dos grupos: "Jóvenes" y "Adultos". Posteriormente voy a usar porcentajes, así podré comparar bien qué prefiere cada grupo, sin importar si hay más gente en un grupo que en otro. Los pasos a seguir han sido estos:


1. Primero se crea una columna nueva que se llame Grupo Etario. Con la función .loc, voy a separar a los clientes en dos categorías: los que tienen "40 o menos" y los que son "Mayores a 40", de esta manera tengo solo dos grupos fáciles de comparar.
2. Voy a usar un *groupby* para ver cuánta gente quedó en cada grupo y cuánto gastan en promedio. Esto me sirve para saber si un grupo gasta mucho más que el otro.
3. Para ver qué productos prefieren, voy a calcular el porcentaje de ventas de cada categoría (Ropa, Electrónica, Belleza).
4. Hacer la gráfica comparativa: Al final, uso una gráfica de barras agrupadas para entender los resultados en todas sus variantes.


In [10]:
# Creamos una nueva columna clasificadora
df['Grupo Etario'] = 'Mayor a 40'
df.loc[df['Age'] <= 40, 'Grupo Etario'] = '40 o Menor'

# Vemos que la columna se creó correctamente
print("Total de clientes por grupo:")
print(df['Grupo Etario'].value_counts())

print()

# 2. Comparamos el Ticket Promedio usando nuestra nueva columna
ticket_edades = df.groupby('Grupo Etario')['Total Amount'].mean().round(2)
print("--- Ticket Promedio por Edad ---")
print(ticket_edades)
print()

Total de clientes por grupo:
Grupo Etario
Mayor a 40    534
40 o Menor    466
Name: count, dtype: int64

--- Ticket Promedio por Edad ---
Grupo Etario
40 o Menor    491.19
Mayor a 40    425.29
Name: Total Amount, dtype: float64



In [11]:
# 2. Calculamos los porcentajes en una sola tabla
pref_edades = df.groupby('Grupo Etario')['Product Category'].value_counts(normalize=True).reset_index(name='Porcentaje')

# 3. Multiplicamos por 100 para hacerlo con formato de porcentaje
pref_edades['Porcentaje'] = (pref_edades['Porcentaje'] * 100).round(2)

# Vemos cómo quedó nuestra tabla lista para graficar
pref_edades

,Grupo Etario,Product Category,Porcentaje
0,40 o Menor,Clothing,33.69
1,40 o Menor,Electronics,33.48
2,40 o Menor,Beauty,32.83
3,Mayor a 40,Clothing,36.33
4,Mayor a 40,Electronics,34.83
5,Mayor a 40,Beauty,28.84


In [12]:
# 4. Hacemos el gráfico 
fig2 = px.bar(
    pref_edades,
    x='Product Category',
    y='Porcentaje',
    color='Grupo Etario',
    barmode='group',
    text_auto='.2f',
    title='Preferencias por categoría según grupo de edad',
    labels={'Product Category': 'Categoría'}
)

fig2.show()

 #### La edad tiene un impacto directo en las preferencias de consumo. Los clientes de 40 años o menos son consumidores hacen sus compras de forma equitativa entre Ropa (33.69%), Electrónica (33.48%) y Belleza (32.83%). Mientras que a los clientes de mas de 40 años, pierden interés en la categoría de Belleza (28.84%) y se enfocan en comprar Ropa (36.33%) y Electrónica (34.83%).

# Objetivo 3: Evaluar si la edad del cliente incrementa la tendencia a comprar artículos premium y el gasto total


**Justificación Metodológica:** 
En este punto quiero investigar si las personas de mayor tienden a comprar cosas más caras. Los pasos realizados son los siguientes:

1. Utilizar *.loc* para separar a los clientes en cuatro rangos: los de 18-25, 26-35, 36-50 y los de más de 51 años. Así puedo ver mejor el cambio de comportamiento según pasan los años.
2. Se creó una columna nueva llamada Tipo de Producto, la cual tiene como condición que si el precio por unidad es mayor a 300, se marca como "Premium" y si cuesta menos, se marca como "Económico". Esto me ayuda a separar las compras de lujo de las normales.
3. Agrupé los datos por el rango de edad y el tipo de producto. Calculé el promedio de cuánto gastan y qué porcentaje de sus compras son del tipo "Premium". Usé porcentajes porque me interesa saber qué tan seguido eligen lo costoso dentro de su propio grupo.
4. Al final, usé una gráfica de barras y presenté la conclusion de los datos.

In [13]:
# CREAMOS RANGOS DE EDAD
df['Rango_Edad'] = 'Por asignar'

df.loc[df['Age'] <= 25, 'Rango_Edad'] = '18-25'
df.loc[(df['Age'] > 25) & (df['Age'] <= 35), 'Rango_Edad'] = '26-35'
df.loc[(df['Age'] > 35) & (df['Age'] <= 50), 'Rango_Edad'] = '36-50'
df.loc[df['Age'] > 50, 'Rango_Edad'] = '51+'


df['Tipo de Producto'] = 'Economico'
df.loc[df['Price per Unit'] > 300, 'Tipo de Producto'] = 'Premium'

analisis = df.groupby(['Rango_Edad', 'Tipo de Producto']).agg({
    'Quantity': 'mean',
    'Total Amount': 'mean',
    'Price per Unit': 'mean'
})

print(analisis)

porcentaje_tipo = (df.groupby('Rango_Edad')['Tipo de Producto']
                     .value_counts(normalize=True)
                     .reset_index(name='Porcentaje'))

porcentaje_tipo['Porcentaje'] = (porcentaje_tipo['Porcentaje'] * 100).round(2)

porcentaje_tipo

                             Quantity  Total Amount  Price per Unit
Rango_Edad Tipo de Producto                                        
18-25      Economico         2.393939    276.893939      105.568182
           Premium           2.594595   1297.297297      500.000000
26-35      Economico         2.664634    286.463415      103.750000
           Premium           2.512195   1256.097561      500.000000
36-50      Economico         2.488281    256.484375      101.933594
           Premium           2.596491   1298.245614      500.000000
51+        Economico         2.526104    238.192771       93.755020
           Premium           2.312500   1156.250000      500.000000


,Rango_Edad,Tipo de Producto,Porcentaje
0,18-25,Economico,78.11
1,18-25,Premium,21.89
2,26-35,Economico,80.00
3,26-35,Premium,20.00
4,36-50,Economico,81.79
5,36-50,Premium,18.21
6,51+,Economico,79.55
7,51+,Premium,20.45


In [14]:
# Usamos px.bar
fig3 = px.bar(
    porcentaje_tipo,                   
    x='Rango_Edad',
    y='Porcentaje',
    color='Tipo de Producto',          
    text_auto='.2f',                   
    title='Proporción de compras: Económico vs Premium por Edad',
    color_discrete_map={'Economico': 'blue', 'Premium': 'red'},
    labels={'Rango_Edad': 'Rango de Edad', 'Tipo de Producto': 'Tipo de Producto'}
)

fig3.show()

##### Los datos demuestran que el segmento más joven (18-25 años) es el que mayor porcentaje de sus compras destina a productos Premium (21.89%). Tambien se observa que en clientes de mayor edad tienen menor preferencia a los articulos premium, siendo el grupo de adultos de 36 a 50 años el que menos artículos de lujo compra (solo 18.21% de Premium frente a un 81.79% de Económicos).


# Objetivo 4: Identificar qué rango de edad concentra el mayor volumen de ingresos acumulados

**Justificación Metodológica:** 
Hemos decidido sumar todo el dinero para ver qué grupo es el que más le aporta a la tienda. Hemos separado este proceso en dos partes:

1. Usamos un *groupby* con los rangos de edad que creé antes, pero ahora le pedí que me dé la suma total (.sum()) de la columna Total Amount. Esto me da una tabla con datos de dinero por cada bando.
2. Con los totales conseguidos se realizó una gráfica de barras y se procedió a explicar los resultados.


In [15]:
analisis = df.groupby('Rango_Edad', as_index=False)['Total Amount'].sum()
analisis


,Rango_Edad,Total Amount
0,18-25,84550
1,26-35,98480
2,36-50,139660
3,51+,133310


In [16]:
fig4_alt = px.bar(
    analisis, 
    x='Rango_Edad', 
    y='Total Amount', 
    text_auto='.2s',              
    color='Rango_Edad',
    title='Ingresos Acumulados por Rango de Edad'
)

fig4_alt.show()

### Los datos demuestran que el segmento de adultos entre 36 y 50 años aportó el mayor volumen de facturación total ($139,660), seguido por el segmento de personas mayores a 51 años. Tambien podemos apreciar que las personas mas jovenes presentan un menor gasto total. De esta manera identificamos que existe una correlación entre el gasto total y la edad del cliente. 

# Objetivo 5: Evaluar si hombres y mujeres presentan diferencias en el volumen de transacciones y en los ingresos generados


**Justificación Metodológica:** 
Aquí hemos decidido mezclar el género con los rangos de edad para ver quiénes son los mejores clientes.

1. Usé el *groupby* pero esta vez metí dos cosas: el Rango_Edad y el Gender. Con la función .agg(), le pedí al código que contara cuántas compras se hicieron y que sumara todo el dinero gastado.
2. Hice una gráfica de barras agrupada que me permite ver, por cada edad, una barra rosa para las mujeres y una azul para los hombres.

In [17]:
analisis_segmento = df.groupby(['Rango_Edad', 'Gender'], as_index=False).agg(
    Transacciones=('Total Amount', 'count'),
    Ingresos_Totales=('Total Amount', 'sum')
)

print(analisis_segmento)

  Rango_Edad  Gender  Transacciones  Ingresos_Totales
0      18-25  Female             81             39470
1      18-25    Male             88             45080
2      26-35  Female            107             55115
3      26-35    Male             98             43365
4      36-50  Female            164             70790
5      36-50    Male            149             68870
6        51+  Female            158             67465
7        51+    Male            155             65845


In [18]:
fig5_2 = px.bar(
    analisis_segmento,
    x='Rango_Edad',
    y='Ingresos_Totales',
    color='Gender',
    barmode='group',
    title='Ingresos totales por segmento',
    color_discrete_map={
        'Female': 'Pink',
        'Male': 'Blue'
    }
)

fig5_2.show()


### Podemos observar que el segmento de mujeres en el rango de 36-50 años son quienes hacen más transacciones (164) y quienes más dinero dejan ($70,790).

# STREAMLIT

**Justificación Metodológica:** 
Para terminar el proyecto, voy a se va a juntar todo lo calculado antes en una sola aplicación interactiva usando Streamlit. Esto nos permite que no sea solo un reporte estático, sino una página donde cualquiera pueda mover los filtros y ver cómo cambian los resultados. Se trabajó este proceso en distintas etapas:


1. Se realizó una función que carga el archivo limpio y prepara todas las columnas (los rangos de edad y artículos Premium). Le puse una etiqueta que dice @st.cache_data; esto es para que la página cargue rápido y no tenga que leer todo el archivo desde cero cada vez que se haga un clic.
2. Usé st.columns para dividir la pantalla en 4 partes y poner tarjetas con los números más importantes: el total de ventas, cuánto dinero entró en total, el ticket promedio y cuál fue la categoría que más vendió.
3. Se utilizó *st.tabs* para que la página no se viera amontonada con tanta información. Así separé el análisis en 6 pestañas diferentes.
4. En cada pestaña puse botones de selección (selectbox o multiselect).
5. Gráficas que cambian solas: Metí todas las gráficas de Plotly que hice en los objetivos anteriores.
6. Usé st.info para poner un pequeño texto explicando qué se ve en cada gráfica y st.expander para esconder las conclusiones finales. Así, si alguien tiene curiosidad de saber qué significa el gráfico, solo tiene que darle clic para que se despliegue el texto.

In [ ]:
%%writefile app_retail.py
import streamlit as st
import pandas as pd
import plotly.express as px

# 1. Configuración Inicial y Carga de Datos
st.set_page_config(page_title="Dashboard Retail", layout="wide")
st.title("Retail: Comportamiento del Cliente y Ventas")
st.write("Este proyecto presenta un análisis exploratorio y descriptivo de una base de datos del sector retail. El objetivo principal es examinar el impacto de las características demográficas de los clientes sobre el comportamiento de compra, la preferencia de categorías de productos y el volumen total de facturación.")


  # Creación de columnas y agrupación de datos
@st.cache_data
def columnasyrangos():
    df = pd.read_csv('Retail_Limpio.csv')

    # Objetivo 2: Preparación de columna Grupo Etario
    df['Grupo Etario'] = 'Mayor a 40'
    df.loc[df['Age'] <= 40, 'Grupo Etario'] = '40 o Menor'

    # Objetivo 3: Preparación de Rangos de Edad y Tipo de Producto
    df.loc[df['Age'] <= 25, 'Rango_Edad'] = '18-25'
    df.loc[(df['Age'] > 25) & (df['Age'] <= 35), 'Rango_Edad'] = '26-35'
    df.loc[(df['Age'] > 35) & (df['Age'] <= 50), 'Rango_Edad'] = '36-50'
    df.loc[df['Age'] > 50, 'Rango_Edad'] = '51+'
    
    df['Tipo de Producto'] = 'Económico'
    df.loc[df['Price per Unit'] > 300, 'Tipo de Producto'] = 'Premium'
    
    
    df['Gender'] = df['Gender'].replace({'Male': 'Masculino', 'Female': 'Femenino'})
    df['Product Category'] = df['Product Category'].replace({
        'Beauty': 'Belleza', 
        'Clothing': 'Ropa', 
        'Electronics': 'Electrónica'
    })

    return df

df = columnasyrangos()



# Creamos las 4 columnas
col1, col2, col3, col4 = st.columns(4)

# Tarjeta 1: Total de Transacciones
with col1:
    with st.container(border=True):
        total_transacciones = len(df)
        st.metric(label="Total de Transacciones", value=f"{total_transacciones:,}")
        st.caption("Registros válidos analizados")

# Tarjeta 2: Ingresos Totales
with col2:
    with st.container(border=True):
        ingresos_totales = df['Total Amount'].sum()
        st.metric(label="Ingresos Totales", value=f"${ingresos_totales:,.2f}")
        st.caption("Facturación global del periodo")

# Tarjeta 3: Ticket Promedio
with col3:
    with st.container(border=True):
        ticket_promedio = df['Total Amount'].mean()
        st.metric(label="Ticket Promedio", value=f"${ticket_promedio:,.2f}")
        st.caption("Gasto promedio por visita")

# Tarjeta 4: Categoría Top
with col4:
    with st.container(border=True):
        cat_estrella = df.groupby('Product Category')['Total Amount'].sum().idxmax()
        st.metric(label="Categoría Top", value=cat_estrella)
        st.caption("Mayor volumen de ingresos")

st.divider()


tab1, tab2, tab3, tab4, tab5, tab6 = st.tabs(["Tabla de datos", "Ticket Promedio por Género", "Edad y Categoría de Producto", "Tendencia a Productos Premium", "Ingresos Acumulados", "Transacciones vs Ingresos"])

with tab1:
    st.subheader("📋 Tabla de datos")
    # 1. Creamos el Selectbox con las opciones
    filtro_categoria = st.selectbox(
        "Filtrar transacciones por categoría:",
        ["Todas las categorías", "Belleza", "Ropa", "Electrónica"],
        key="filtro_cat"
        )

    if filtro_categoria == "Todas las categorías":
         df_filtrado = df
    else:
     #Filtramos el df original donde la columna coincida con la selección
         df_filtrado = df[df['Product Category'] == filtro_categoria]

    # Mostramos un pequeño texto indicando cuántos registros encontró
    st.caption(f"Mostrando {len(df_filtrado)} transacciones correspondientes a: **{filtro_categoria}**")
    df_Trad = df_filtrado.rename(columns={
        'Date': 'Fecha',
        'Gender': 'Género',
        'Age': 'Edad',
        'Product Category': 'Categoría',
        'Quantity': 'Cantidad',
        'Price per Unit': 'Precio Unitario ($)',
        'Total Amount': 'Monto Total ($)',
        'Rango_Edad': 'Rango de Edad'
    })



    # 4. Mostramos la tabla interactiva TRADUCIDA
    with st.container(border=True):
        st.dataframe(df_Trad, use_container_width=True)


    # -------------------------
    # Expander (información extra)
    # -------------------------
    with st.expander("Mostrar estadísticas descriptivas"):
        st.write(df.describe())
        st.divider()

    # -------------------------
    # # Expander
    # # -------------------------
    # st.header("Expander")

    with st.expander("Bibliografía"):
        st.write("https://plotly.com/python/bar-charts/")
        st.write("https://docs.streamlit.io/develop/tutorials")



with tab2:
# 1. Creamos el menú de selección múltiple (por defecto mostramos ambos)
    generos_seleccionados = st.multiselect(   "Selecciona el género a visualizar:",
         options=["Femenino", "Masculino"],
         default=["Femenino", "Masculino"] )
# 2. Calculamos los datos
    ticket_genero = df.groupby('Gender')['Total Amount'].mean().round(2).reset_index()


# 4. Hacemos el gráfico con la tabla filtrada
    fig1 = px.bar(
        ticket_genero, # Usamos la nueva tabla filtrada
        x='Gender',
        y='Total Amount',
        color='Gender',
        text_auto='.2f',
        title='Gasto Promedio por Género',
        color_discrete_map={'Femenino': 'lightpink', 'Masculino': 'darkblue'},
        labels={'Gender': 'Género', 'Total Amount': 'Ticket Promedio ($)'})
    
    # 3. Filtramos la tabla según lo que el usuario eligió en el menú
    ticket_genero_filtrado = ticket_genero[ticket_genero['Gender'].isin(generos_seleccionados)]

    st.plotly_chart(fig1)

    st.info( "📊 **Observación de datos:** El ticket promedio de compra es prácticamente idéntico entre hombres (\\$455.43) y mujeres (\\$456.55).")
     
    with st.expander("Conclusión"):
        st.success("Tras analizar las transacciones, se concluye que el género del cliente no es un factor que determine o altere el monto promedio gastado por visita.")

    st.divider()

with tab3:
    # --- OBJETIVO 2: Edad y Categoría de Producto ---
    st.header("2. Preferencias de categoría por edad")

    Catogorias_seleccionadas = st.multiselect(   "Selecciona la categoria a visualizar:",
         options=["Ropa", "Electrónica", "Belleza"],
         default=["Ropa", "Electrónica", "Belleza"])

    

    pref_edades = df.groupby('Grupo Etario')['Product Category'].value_counts(normalize=True).reset_index(name='Porcentaje')
    pref_edades['Porcentaje'] = (pref_edades['Porcentaje'] * 100).round(2)
    
    pref_edades_filtrado = pref_edades[pref_edades['Product Category'].isin(Catogorias_seleccionadas)]

    
    fig2 = px.bar(pref_edades_filtrado, x='Product Category', y='Porcentaje', color='Grupo Etario',
                barmode='group', text_auto='.2f', title='Preferencia por Categoría (%)',
                labels={'Product Category': 'Categoría de Producto', 'Porcentaje': 'Porcentaje (%)'} 
    )

    
    st.plotly_chart(fig2)

    st.info("📊 **Observación de datos:** Los clientes de 40 años o menos compran de forma equitativa (Ropa 33.69%, Electrónica 33.48%, Belleza 32.83%). Los mayores de 40 reducen sus compras de Belleza al 28.84%.")

    with st.expander("💡 Ver Conclusión"):
        st.success("La edad influye directamente en las preferencias de consumo. Existe un cambio generacional claro donde el interés por los artículos de belleza disminuye significativamente al superar los 40 años en favor de otras categorías.")


    st.divider()



with tab4:
    # --- OBJETIVO 3: Tendencia a Productos Premium ---
    st.header("3. Tendencia de Consumo Premium por Edad")
    porcentaje_tipo = df.groupby('Rango_Edad')['Tipo de Producto'].value_counts(normalize=True).reset_index(name='Porcentaje')
    porcentaje_tipo['Porcentaje'] = (porcentaje_tipo['Porcentaje'] * 100).round(2)
    
    tipo_prod = st.multiselect(
        "Selecciona el tipo de producto a visualizar:",
        options=["Económico", "Premium"],
        default=["Económico", "Premium"]
        )
    
    porcentaje_tipo_filtrado = porcentaje_tipo[porcentaje_tipo['Tipo de Producto'].isin(tipo_prod)]

    fig3 = px.bar(porcentaje_tipo_filtrado, x='Rango_Edad', y='Porcentaje', color='Tipo de Producto',
                barmode='group', text_auto='.2f', title='Económico vs Premium por Rango',
                labels={'Rango_Edad': 'Rango de Edad', 'Porcentaje': 'Porcentaje (%)', 'Tipo de Producto': 'Gama de Producto'},
                color_discrete_map={'Económico': 'Green', 'Premium': 'darkblue'}
                )


    st.plotly_chart(fig3)

    st.info("📊 **Observación de datos:** El segmento de 18-25 años concentra la mayor proporción de compras Premium (21.89%). Esta cifra disminuye gradualmente con la edad, cayendo al 18.21% en el grupo de 36-50 años.")

    with st.expander("💡 Ver Conclusión"):
        st.success("Se refuta la hipótesis inicial: a mayor edad no existe una mayor tendencia a comprar artículos de lujo. De hecho, el público más joven es el que proporcionalmente destina más compras a la gama Premium.")


    st.divider()


with tab5:
    # --- OBJETIVO 4: Ingresos Acumulados ---
    st.header("4. Distribución de ingresos por edad")
    fig4 = px.pie(df, values='Total Amount', names='Rango_Edad', hole=0.4,
                title='Volumen de Ingresos Acumulados',
                labels={'Rango_Edad': 'Rango de Edad', 'Total Amount': 'Ingresos Acumulados ($)'}
                )

    st.plotly_chart(fig4)

    st.info("📊 **Observación de datos:** El grupo de adultos entre 36 y 50 años generó el mayor volumen de facturación total (\\$139,660), superando ampliamente al segmento de 18-25 años (\\$84,550).")

    with st.expander("💡 Ver Conclusión"):
        st.success("Aunque los jóvenes compran más artículos Premium en proporción, los adultos de 36 a 50 años se consolidan como el verdadero motor financiero del negocio gracias a un mayor volumen operativo de transacciones.")


    st.divider()



with tab6:
    # --- OBJETIVO 5: Transacciones vs Ingresos ---
    st.header("5. Actividad vs Rentabilidad por Segmento")

    transacciones = df.groupby(['Rango_Edad', 'Gender']).size().reset_index(name='Cantidad')

    ingresos = df.groupby(['Rango_Edad', 'Gender'])['Total Amount'].sum().reset_index()

    generos_seleccionados = st.multiselect(
        "Selecciona el género a visualizar en ambos gráficos:",
        options=["Femenino", "Masculino"],
        default=["Femenino", "Masculino"]
    )


    trans_filtradas = transacciones[transacciones['Gender'].isin(generos_seleccionados)]
    ingresos_filtrados = ingresos[ingresos['Gender'].isin(generos_seleccionados)]

    col1, col2 = st.columns(2)

    with col1:
        fig5_1 = px.bar(trans_filtradas, x='Rango_Edad', y='Cantidad', color='Gender',
                        barmode='group', title='Volumen de Transacciones',
                        labels={'Rango_Edad': 'Rango de Edad', 'Cantidad': 'N.° de Transacciones', 'Gender': 'Género'}
                        ) 
        st.plotly_chart(fig5_1, use_container_width=True)

    with col2:
        
        fig5_2 = px.bar(ingresos_filtrados, x='Rango_Edad', y='Total Amount', color='Gender',
                        barmode='group', title='Ingresos Totales ($)',
                        labels={'Rango_Edad': 'Rango de Edad', 'Total Amount': 'Ingresos ($)', 'Gender': 'Género'}
                )
        st.plotly_chart(fig5_2)



    st.info("📊 **Observación de datos:** Las mujeres de 36 a 50 años lideran de forma absoluta en número de transacciones (164) e ingresos generados (\\$70,790). En contraste, en el segmento joven (18-25) los hombres compran más.")

    with st.expander("💡 Ver Conclusión"):
        st.success("Existe una correlación directa entre la cantidad de visitas y la rentabilidad. El perfil demográfico clave de la tienda es la mujer adulta, mientras que en la demografía juvenil el consumo es impulsado principalmente por los hombres.")

    st.divider()


Overwriting app_retail.py


In [20]:
!streamlit run app_retail.py

<unknown>:157: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
<unknown>:240: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Hp\AppData\Local\Programs\Python\Python314\Scripts\streamlit.exe\__main__.py", line 6, in <module>
    sys.exit(main())
             ~~~~^^
  File "c:\Users\Hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\click\core.py", line 1485, in __call__
    return self.main(*args, **kwargs)
           ~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "c:\Users\Hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\click\core.py", line 1406, in main
    rv = self.invoke(ctx)
  File "c:\Users\Hp\AppData\Local\Programs\Pyth